install libs

In [30]:
try:
	import plotly
	from sklearn.preprocessing import StandardScaler
	from sklearn.decomposition import TruncatedSVD
except ImportError:
	%pip install -q scikit-learn plotly
	from sklearn.preprocessing import StandardScaler
	from sklearn.decomposition import TruncatedSVD

### Read data from parquet files


In [31]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt


# Define the directory containing the Parquet files
data_dir = os.path.join('data', 'iam_dataset')

# Read each Parquet file into a separate DataFrame
account_entitlements_df = pd.read_parquet(os.path.join(data_dir, 'account_entitlements.parquet'))
accounts_df = pd.read_parquet(os.path.join(data_dir, 'accounts.parquet'))
applications_df = pd.read_parquet(os.path.join(data_dir, 'applications.parquet'))
entitlement_group_assignments_df = pd.read_parquet(os.path.join(data_dir, 'entitlement_group_assignments.parquet'))
entitlement_group_relations_df = pd.read_parquet(os.path.join(data_dir, 'entitlement_group_relations.parquet'))
group_entitlements_df = pd.read_parquet(os.path.join(data_dir, 'group_entitlements.parquet'))
entitlement_groups_df = pd.read_parquet(os.path.join(data_dir, 'entitlement_groups.parquet'))
entitlements_df = pd.read_parquet(os.path.join(data_dir, 'entitlements.parquet'))
identities_df = pd.read_parquet(os.path.join(data_dir, 'identities.parquet'))
resources_df = pd.read_parquet(os.path.join(data_dir, 'resources.parquet'))

# Print the shapes of the DataFrames
print(f"Shape of account_entitlements_df: {account_entitlements_df.shape}")
print(f"Shape of accounts_df: {accounts_df.shape}")
print(f"Shape of applications_df: {applications_df.shape}")
print(f"Shape of entitlement_group_assignments_df: {entitlement_group_assignments_df.shape}")
print(f"Shape of entitlement_group_relations_df: {entitlement_group_relations_df.shape}")
print(f"Shape of entitlement_groups_df: {entitlement_groups_df.shape}")
print(f"Shape of entitlements_df: {entitlements_df.shape}")
print(f"Shape of identities_df: {identities_df.shape}")
print(f"Shape of resources_df: {resources_df.shape}")

Shape of account_entitlements_df: (972420, 5)
Shape of accounts_df: (314631, 6)
Shape of applications_df: (1000, 4)
Shape of entitlement_group_assignments_df: (170010, 3)
Shape of entitlement_group_relations_df: (571, 2)
Shape of entitlement_groups_df: (711, 5)
Shape of entitlements_df: (22991, 5)
Shape of identities_df: (50000, 8)
Shape of resources_df: (2100, 5)


## EDA

### Distribution of accounts per user

In [32]:
# Group by 'identity_id' and count the number of accounts for each user
accounts_per_user = accounts_df.groupby('identity_id').size()

# Get the distribution of the number of accounts
account_distribution = accounts_per_user.value_counts()

print("Distribution of accounts per user:")
print(account_distribution)

Distribution of accounts per user:
6     9966
5     9911
7     9010
4     8772
9     7904
8     4192
10     245
Name: count, dtype: int64


### Distribution of entitlements per account

In [33]:
# --- Analysis for account_entitlements ---

# Group by 'account_id' and count the number of entitlements for each account
entitlements_per_account = account_entitlements_df.groupby('account_id').size()

# Get the distribution of the number of entitlements
entitlement_distribution = entitlements_per_account.value_counts()

print("Distribution of entitlements per account:")
print(entitlement_distribution)

# Create a bar plot of the entitlement distribution
import plotly.express as px
fig = px.bar(
    x=entitlement_distribution.index, 
    y=entitlement_distribution.values, 
    labels={'x':'Number of Entitlements', 'y':'Number of Accounts'},
    title='Distribution of Entitlements per Account'
)
fig.update_layout(xaxis_type='category')
fig.show()



Distribution of entitlements per account:
1     240438
2      24193
20      9012
13      7010
5       5027
12      4041
17      4025
9       3968
14      3964
19      3031
10      1967
15      1962
18      1943
6       1055
8       1026
16       998
7        971
Name: count, dtype: int64


### Distribution of entitlement groups per identity:

In [34]:
# --- Analysis for entitlement_group_assignments ---

# Group by 'identity_id' and count the number of entitlement groups for each identity
entitlement_groups_per_identity = entitlement_group_assignments_df.groupby('identity_id').size()

# Get the distribution of the number of entitlement groups
entitlement_group_distribution = entitlement_groups_per_identity.value_counts()

print("\nDistribution of entitlement groups per identity:")
print(entitlement_group_distribution)

# Create a bar plot of the entitlement group distribution
import plotly.express as px
fig = px.bar(
    x=entitlement_group_distribution.index, 
    y=entitlement_group_distribution.values, 
    labels={'x':'Number of Entitlement Groups', 'y':'Number of Identities'},
    title='Distribution of Entitlement Groups per Identity'
)
fig.update_layout(xaxis_type='category')
fig.show()



Distribution of entitlement groups per identity:
3    31981
4    16028
5     1991
Name: count, dtype: int64


### Columns in each entity

In [35]:
print("applications_df columns:", applications_df.columns)
print("resources_df columns:", resources_df.columns)
print("entitlements_df columns:", entitlements_df.columns)
print("account_entitlements_df columns:", account_entitlements_df.columns)
print("accounts_df columns:", accounts_df.columns)

applications_df columns: Index(['app_id', 'app_name', 'business_criticality', 'app_owner_id'], dtype='str')
resources_df columns: Index(['resource_id', 'app_id', 'iga_source_name', 'connection_type',
       'resource_type'],
      dtype='str')
entitlements_df columns: Index(['entitlement_id', 'resource_id', 'entitlement_name', 'is_requestable',
       'owner_id'],
      dtype='str')
account_entitlements_df columns: Index(['identity_id', 'account_id', 'entitlement_id', 'grant_date',
       'assignment_type'],
      dtype='str')
accounts_df columns: Index(['identity_id', 'resource_id', 'account_id', 'account_name',
       'is_privileged', 'status'],
      dtype='str')


### Distribution of resources per application

In [36]:
# --- Analysis for application to resource mapping ---

# Group by 'app_id' and count the number of resources for each application
resources_per_app = resources_df.groupby('app_id').size()

# Get the distribution of the number of resources
resource_distribution = resources_per_app.value_counts().sort_index()

print("Distribution of resources per application:")
print(resource_distribution)

# Create a bar plot of the resource distribution
import plotly.express as px
fig = px.bar(
    x=resource_distribution.index, 
    y=resource_distribution.values, 
    labels={'x':'Number of Resources', 'y':'Number of Applications'},
    title='Distribution of Resources per Application'
)
fig.update_layout(xaxis_type='category')
fig.show()


Distribution of resources per application:
1     640
2     203
3      28
4      48
5      20
6       9
7       2
8       6
9       1
10      8
11      4
12      5
13      6
14      9
15     11
Name: count, dtype: int64


### Distribution of entitlements per resource

In [37]:
# --- Analysis for resource to entitlement mapping ---

# Group by 'resource_id' and count the number of entitlements for each resource
entitlements_per_resource = entitlements_df.groupby('resource_id').size()

# Get the distribution of the number of entitlements per resource
resource_entitlement_dist = entitlements_per_resource.value_counts().sort_index()

print("Distribution of entitlements per resource:")
print(resource_entitlement_dist)

# Create a bar plot of the resource to entitlement distribution
import plotly.express as px
fig = px.bar(
    x=resource_entitlement_dist.index, 
    y=resource_entitlement_dist.values, 
    labels={'x':'Number of Entitlements', 'y':'Number of Resources'},
    title='Distribution of Entitlements per Resource'
)
fig.update_layout(xaxis_type='category')
fig.show()


Distribution of entitlements per resource:
1      446
2      463
3      470
4       89
5       74
      ... 
454      1
459      1
460      1
469      1
474      1
Name: count, Length: 75, dtype: int64


### Distribution of accounts per entitlement:

In [38]:
# --- Analysis for entitlement to account mapping ---

# Group by 'entitlement_id' and count the number of unique accounts for each entitlement
accounts_per_entitlement = account_entitlements_df.groupby('entitlement_id')['account_id'].nunique()

# Get the distribution of the number of accounts per entitlement
account_dist_per_entitlement = accounts_per_entitlement.value_counts().sort_index()

print("Distribution of accounts per entitlement:")
print(account_dist_per_entitlement)

# Create a bar plot of the distribution
import plotly.express as px
fig = px.bar(
    x=account_dist_per_entitlement.index, 
    y=account_dist_per_entitlement.values, 
    labels={'x':'Number of Accounts', 'y':'Number of Entitlements'},
    title='Distribution of Accounts per Entitlement'
)
fig.update_layout(xaxis_type='category')
fig.show()


Distribution of accounts per entitlement:
account_id
1       1077
2        128
3         22
4          3
939       12
        ... 
2018       1
2027       2
2032       1
2043       1
2947       1
Name: count, Length: 94, dtype: int64


### Distribution of assignment counts per Entitlement Group

In [39]:
# User–entitlement mapping: merge account_entitlements with entitlements to get entitlement_name
user_entitlements_df = (
    account_entitlements_df[['identity_id', 'entitlement_id']]
    .merge(entitlements_df[['entitlement_id', 'entitlement_name']], on='entitlement_id', how='left')
    [['identity_id', 'entitlement_name']]
    .drop_duplicates()
)
print("user_entitlements_df shape:", user_entitlements_df.shape)
user_entitlements_df.head()

user_entitlements_df shape: (972419, 2)


,identity_id,entitlement_name
0,2e420c0e-7360-4c03-abe2-184c794f6254,Ent_TREE_9206
1,2e420c0e-7360-4c03-abe2-184c794f6254,Ent_AUTHOR_1911
2,2e420c0e-7360-4c03-abe2-184c794f6254,Ent_HOME_2386
3,2e420c0e-7360-4c03-abe2-184c794f6254,Ent_NIGHT_5978
4,2e420c0e-7360-4c03-abe2-184c794f6254,Ent_HEALTH_1542


In [40]:
# --- Analysis for entitlement group assignment distribution ---

# Group by 'ent_group_id' and count the number of identities assigned to each group
group_size = entitlement_group_assignments_df.groupby('ent_group_id').size()

# Get the distribution of the number of assignments
group_dist = group_size.value_counts().sort_index()

print("Distribution of assignment counts per Entitlement Group:")
print(group_dist)

# Create a bar plot of the distribution
import plotly.express as px
fig = px.bar(
    x=group_dist.index.astype(int), 
    y=group_dist.values, 
    labels={'x':'Number of Identities Assigned', 'y':'Number of Groups'},
    title='Distribution of Assignments per Entitlement Group'
)
fig.update_layout(xaxis_type='category')
fig.show()


Distribution of assignment counts per Entitlement Group:
31       1
34       1
35       1
38       1
39       3
        ..
5005     1
5018     1
5053     1
5084     1
50000    1
Name: count, Length: 140, dtype: int64


## Feature Engineering

The following cells build a mathematical feature matrix from `df_identities`, `df_entitlements`, and `df_user_entitlements` (identity_id, entitlement_name, risk_level) for unsupervised peer-group clustering.

In [41]:
# --- Step 1: Contextual Feature Engineering (HR Data) ---
# Isolate active users (assume column 'status' with value 'Active'; adjust if your schema differs)
active_identities_df = identities_df[identities_df['status'] == 'Active'].copy()

# One-Hot Encode department, job_title, location; set identity_id as index
hr_features_df = pd.get_dummies(
    active_identities_df[['identity_id', 'department', 'job_title', 'location']],
    columns=['department', 'job_title', 'location'],
    dtype=int
).set_index('identity_id')

print("Step 1 — HR features shape:", hr_features_df.shape)
hr_features_df.head()

Step 1 — HR features shape: (45151, 23577)


,department_Customer Support,department_Engineering,department_Finance,department_HR,department_IT,department_Legal,department_Marketing,department_Operations,department_Product,department_Sales,...,location_Zimmermanshire,location_Zimmermanstad,location_Zimmermantown,location_Zoeberg,location_Zoeburgh,location_Zoeside,location_Zunigamouth,location_Zunigaside,location_Zunigaton,location_Zunigatown
identity_id,,,,,,,,,,,,,,,,,,,,,
2e420c0e-7360-4c03-abe2-184c794f6254,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
819ac6a0-8504-4823-802f-92fd6074096e,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1cb6f71e-27d6-430d-88c9-425d7d66ceb6,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5e73a201-388c-4860-944d-6a2a2e54bb6c,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7621e1ff-5766-406e-91c5-1579e76f376c,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [42]:
# --- Step 2: Behavioral Feature Engineering (User–Entitlement Sparse Matrix) ---
# Build binary User-Entitlement matrix: 1 if user holds entitlement, 0 otherwise
access_features_df = user_entitlements_df.assign(has_entitlement=1).pivot_table(
    index='identity_id',
    columns='entitlement_name',
    values='has_entitlement',
    aggfunc='max',
    fill_value=0
)

print("Step 2 — Access matrix shape:", access_features_df.shape)
access_features_df.iloc[:5, :5]

Step 2 — Access matrix shape: (50000, 2170)


entitlement_name,Ent_ABILITY_4636,Ent_ABILITY_5914,Ent_ABILITY_9126,Ent_ABLE_9194,Ent_ABOUT_6523
identity_id,,,,,
0001c3b1-9a75-41f7-812a-32389cf9c900,0,0,0,0,0
0001d551-5ffa-4556-990d-d9f9d7062fce,0,0,0,0,0
000218b1-bf85-4b93-9643-c1fe5ffb9033,0,0,0,0,0
00024dd4-04d4-4157-90c4-2840c75a40cc,0,0,0,0,0
0002b25b-e003-44ef-9ae6-199d7cb1b0cb,0,0,0,0,0


In [43]:
# --- Step 3: Aggregate Feature Creation ---
total_entitlements_count = account_entitlements_df.groupby('identity_id').size()

high_risk_entitlements_count = (
    account_entitlements_df
    .loc[account_entitlements_df['assignment_type'] == 'Adhoc_Anomaly']
    .groupby('identity_id')
    .size()
)

agg_features_df = pd.DataFrame({
    'total_entitlements_count': total_entitlements_count,
    'high_risk_entitlements_count': high_risk_entitlements_count
}).fillna(0).astype(int)

print("Step 3 — Aggregate features shape:", agg_features_df.shape)
agg_features_df.head()

Step 3 — Aggregate features shape: (50000, 2)


,total_entitlements_count,high_risk_entitlements_count
identity_id,,
0001c3b1-9a75-41f7-812a-32389cf9c900,25,0
0001d551-5ffa-4556-990d-d9f9d7062fce,17,0
000218b1-bf85-4b93-9643-c1fe5ffb9033,18,0
00024dd4-04d4-4157-90c4-2840c75a40cc,19,0
0002b25b-e003-44ef-9ae6-199d7cb1b0cb,27,0


In [44]:
# --- Step 4: Final Assembly & Scaling ---
try:
	from sklearn.preprocessing import StandardScaler
except ImportError:
	%pip install -q scikit-learn
	from sklearn.preprocessing import StandardScaler

# Concatenate HR, access, and aggregate features on identity_id index; fill NaNs with 0
master_feature_matrix_df = pd.concat(
    [hr_features_df, access_features_df, agg_features_df],
    axis=1,
    join='outer'
).fillna(0)

# Scale only aggregate numerical columns so they don't overpower binary columns
agg_cols = ['total_entitlements_count', 'high_risk_entitlements_count']
scaler = StandardScaler()
master_feature_matrix_df[agg_cols] = scaler.fit_transform(master_feature_matrix_df[agg_cols])

print("Step 4 — Master feature matrix shape:", master_feature_matrix_df.shape)
master_feature_matrix_df[agg_cols].describe()

Step 4 — Master feature matrix shape: (50000, 25749)


,total_entitlements_count,high_risk_entitlements_count
count,5.000000e+04,5.000000e+04
mean,1.273293e-16,-4.547474e-18
std,1.000010e+00,1.000010e+00
min,-2.173693e+00,-1.758631e-01
25%,-9.254484e-01,-1.758631e-01
50%,1.147553e-01,-1.758631e-01
75%,7.388774e-01,-1.758631e-01
max,1.987122e+00,5.686241e+00


In [45]:
# --- Step 5: Dimensionality Reduction (TruncatedSVD) ---
try:
	from sklearn.decomposition import TruncatedSVD
except ImportError:
	%pip install -q scikit-learn
	from sklearn.decomposition import TruncatedSVD

n_components = 100
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced = svd.fit_transform(master_feature_matrix_df)

print("Step 5 — Reduced matrix shape:", X_reduced.shape)
print("\nExplained variance ratio (first 100 components):")
print(svd.explained_variance_ratio_)
print("\nCumulative explained variance:", svd.explained_variance_ratio_.sum())

Step 5 — Reduced matrix shape: (50000, 100)

Explained variance ratio (first 100 components):
[5.95156081e-02 4.21125203e-02 2.65206104e-02 2.58726375e-02
 2.51534449e-02 2.47741497e-02 2.40391894e-02 2.36905836e-02
 2.29213130e-02 1.97401259e-02 2.08490545e-02 2.18148353e-02
 2.16346638e-02 1.23350827e-02 1.94280488e-02 1.89590632e-02
 1.85998155e-02 1.83233241e-02 1.98751858e-02 1.95496414e-02
 1.91534531e-02 1.91044116e-02 1.85707682e-02 1.85001975e-02
 1.83098998e-02 1.79790574e-02 1.78067555e-02 1.76179688e-02
 1.73944978e-02 1.71276008e-02 1.68969621e-02 1.67701490e-02
 1.63577734e-02 1.60119456e-02 1.58510964e-02 1.57762620e-02
 1.57567273e-02 1.54910568e-02 1.51359750e-02 1.40025968e-02
 1.34717971e-02 1.30985790e-02 1.30178264e-02 1.28348300e-02
 1.24716286e-02 1.19829450e-02 1.15096557e-02 1.09386044e-02
 1.03567865e-02 1.02275081e-02 9.03435060e-03 3.72885645e-04
 3.71642652e-04 3.70670836e-04 3.57937905e-04 3.47503516e-04
 3.45874969e-04 3.41605682e-04 3.37933930e-04 3.2624

# Modellling

In [46]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

### Step 1: K-Means Clustering (Discovering Peer Groups)

In [47]:
# --- Find optimal k using Silhouette Score ---
k_values = list(range(5, 51, 5))
silhouette_scores = []

for k in k_values:
    # Set n_init to suppress warnings and ensure consistency
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_reduced)
    score = silhouette_score(X_reduced, labels)
    silhouette_scores.append(score)

# Plot the Silhouette Scores
import plotly.express as px
fig = px.line(
    x=k_values, 
    y=silhouette_scores, 
    markers=True, 
    labels={'x':'Number of Clusters (k)', 'y':'Silhouette Score'},
    title='Silhouette Score vs. Number of Clusters (k)'
)
fig.update_traces(line=dict(dash='dash', color='blue'))
fig.show()

# --- Fit optimal/default model ---
# Once you review the plot, you could dynamically change this, but we'll default to 20
optimal_k = 20
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans_model.fit_predict(X_reduced)

# Create a results DataFrame mapped to identity_id 
# (Assuming the row order of X_reduced aligns with master_feature_matrix_df.index)
df_results = pd.DataFrame(index=master_feature_matrix_df.index)
df_results['kmeans_peer_group'] = kmeans_labels

### Step 2: DBSCAN Clustering (Hunting Anomalies)

In [48]:
# Instantiate and fit DBSCAN model
# Note: Both eps and min_samples will likely need tuning based on the density / scale 
# of your specific X_reduced matrix. If too many points are flagged as anomalies (or none), 
# try adjusting these hyperparameters first.
dbscan_model = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan_model.fit_predict(X_reduced)

# Extract labels and add them to our tracking dataframe
df_results['dbscan_label'] = dbscan_labels

# Explicitly flag users with a dbscan label of -1 as anomalies
df_results['is_anomaly'] = df_results['dbscan_label'] == -1

### Step 3: Mapping Results Back to Business Context

In [49]:
# Merge the clustering results back into the original dataframe
# Since df_results was created using master_feature_matrix_df.index, we can use a direct join
df_identities_enriched = identities_df.set_index('identity_id').join(df_results, how='inner')

# Print the total count of anomalies detected by DBSCAN
total_anomalies = df_identities_enriched['is_anomaly'].sum()
print(f'Total anomalies detected by DBSCAN: {total_anomalies}')

# Quick sanity check visualization of the enriched dataframe
display(df_identities_enriched.head())

Total anomalies detected by DBSCAN: 98


,identity_type,status,location,cost_center,department,job_title,manager_id,kmeans_peer_group,dbscan_label,is_anomaly
identity_id,,,,,,,,,,
2e420c0e-7360-4c03-abe2-184c794f6254,Human,Active,North Judithbury,CC-3128,Finance,Financial Analyst,cf45e01b-eb4f-4d52-95ff-9741a6de361e,3,0,False
819ac6a0-8504-4823-802f-92fd6074096e,Service Account,Active,East Jill,CC-6874,Operations,Project Manager,b33c0b2a-26b1-4c29-85a0-89e3450bc262,11,1,False
1cb6f71e-27d6-430d-88c9-425d7d66ceb6,Human,Active,New Roberttown,CC-1479,Product,Head of Product,d60c9086-5da9-458f-bd93-9d938044f67d,8,2,False
5e73a201-388c-4860-944d-6a2a2e54bb6c,Human,Active,East Jessetown,CC-5725,Engineering,Engineering Manager,828c56cb-24a6-4cc9-aefc-fb7963638be6,1,3,False
7621e1ff-5766-406e-91c5-1579e76f376c,Human,Active,Lake Debra,CC-5715,IT,SysAdmin,04340424-bcb1-469e-a1e9-c7485b1610c2,2,4,False


### Step 4: Visualization (The Cluster Map)

In [50]:
import plotly.graph_objects as go
import numpy as np

# Create a boolean mask for anomalies
anomaly_mask = df_results['is_anomaly']

# Extract coordinates specifically for the identified anomalies
x_anomalies = X_reduced[anomaly_mask, 0]
y_anomalies = X_reduced[anomaly_mask, 1]

fig = go.Figure()

# Scatter plot of the general clusters
fig.add_trace(go.Scatter(
    x=X_reduced[~anomaly_mask, 0], 
    y=X_reduced[~anomaly_mask, 1], 
    mode='markers',
    marker=dict(
        color=df_results[~anomaly_mask]['kmeans_peer_group'], 
        colorscale='Viridis', 
        opacity=0.5,
        size=8,
        showscale=True,  # To show the legend/colorbar for clusters
        colorbar=dict(title="KMeans Group", x=1.13)
    ),
    name='Cluster Member',
    hoverinfo='text',
    text=df_results[~anomaly_mask]['kmeans_peer_group'].apply(lambda c: f'Cluster: {c}')
))

# Overlay the DBSCAN anomalies as distinct, large red 'X' markers
fig.add_trace(go.Scatter(
    x=x_anomalies, 
    y=y_anomalies, 
    mode='markers', 
    marker=dict(
        color='red', 
        symbol='x',
        size=12, 
        line=dict(color='black', width=1)
    ),
    name='DBSCAN Anomaly',
    hoverinfo='text',
    text=['Anomaly'] * len(x_anomalies)
))

fig.update_layout(
    title='IAM Peer Group Clusters and Anomalies (First 2 Components)',
    xaxis_title='Component 1',
    yaxis_title='Component 2',
    width=1000,
    height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02, title='Legend')
)

fig.show()


# Phase 5: Evaluation

### Step 1: Cluster Profiling (Peer Group Analysis)
We group the identities by their assigned KMeans cluster to understand the composition of each peer group.

In [51]:
# Group by kmeans_peer_group and calculate mode for department and job_title, and count total users
cluster_profiles = df_identities_enriched.groupby('kmeans_peer_group').agg(
    total_users=('status', 'count'),
    dominant_department=('department', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    dominant_job_title=('job_title', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
).reset_index()

print("Cluster Profiles Summary:")
display(cluster_profiles)

# Create a bar chart showing the size of each cluster
import plotly.express as px
fig = px.bar(
    cluster_profiles,
    x='kmeans_peer_group',
    y='total_users',
    color='dominant_department',
    labels={'kmeans_peer_group': 'Cluster ID', 'total_users': 'Number of Users', 'dominant_department': 'Department'},
    title='Cluster Sizes Categorized by Dominant Department'
)
fig.update_layout(xaxis_type='category')
fig.show()

Cluster Profiles Summary:


,kmeans_peer_group,total_users,dominant_department,dominant_job_title
0,0,1035,Marketing,Designer
1,1,2931,Engineering,Engineering Manager
2,2,948,IT,SysAdmin
3,3,1903,Legal,General Counsel
4,4,973,HR,HRBP
5,5,1013,Finance,Finance Manager
6,6,1469,Legal,Finance Manager
7,7,1052,HR,Recruiter
8,8,1899,IT,IT Manager
9,9,966,IT,Network Engineer


### Step 2: Anomaly Deep Dive
We isolate the records flagged as anomalies by DBSCAN to investigate specific areas of risk.

In [52]:
# Filter the DataFrame for anomalies
df_anomalies = df_identities_enriched[df_identities_enriched['is_anomaly'] == True]

# Group anomalies to analyze the risk origins
anomaly_analysis = df_anomalies.groupby(['department', 'job_title']).size().reset_index(name='anomaly_count')
anomaly_analysis = anomaly_analysis.sort_values(by='anomaly_count', ascending=False)

print("Top Anomaly Origins by Department and Job Title:")
display(anomaly_analysis.head(10))

# Bar chart of anomalies by department
anomalies_by_dept = df_anomalies.groupby('department').size().reset_index(name='anomaly_count')
fig = px.bar(
    anomalies_by_dept.sort_values(by='anomaly_count', ascending=False),
    x='department',
    y='anomaly_count',
    color='department',
    labels={'department': 'Department', 'anomaly_count': 'Number of Anomalies'},
    title='Count of Anomalies by Department'
)
fig.update_layout(showlegend=False)
fig.show()

Top Anomaly Origins by Department and Job Title:


,department,job_title,anomaly_count
3,Customer Support,Support Lead,4
6,Engineering,DevOps Engineer,4
5,Engineering,Architect,4
4,Customer Support,Technical Support,4
16,HR,HRBP,4
22,Legal,Contract Manager,4
14,HR,HR Manager,4
11,Finance,Controller,4
32,Operations,Ops Manager,4
39,Sales,Sales Representative,4


### Step 3: Business Impact Summary
Present a high-level overview of our pipeline's findings.

In [53]:
total_users = len(df_identities_enriched)
total_peer_groups = df_identities_enriched['kmeans_peer_group'].nunique()
total_anomalies = df_anomalies.shape[0]

# Getting the top 3 departments with most anomalies
top_3_anomaly_depts = df_anomalies['department'].value_counts().head(3).index.tolist()

print("== IAM Peer Group Analysis: Executive Summary ==")
print(f"Total Identities Processed: {total_users:,}")
print(f"Total Peer Groups Discovered: {total_peer_groups}")
print(f"Total High-Risk Anomalies Flagged: {total_anomalies} ({(total_anomalies/total_users)*100:.2f}% of users)")
print(f"Top 3 Departments Contributing to Anomalies: {', '.join(top_3_anomaly_depts)}")
print("================================================")

== IAM Peer Group Analysis: Executive Summary ==
Total Identities Processed: 50,000
Total Peer Groups Discovered: 20
Total High-Risk Anomalies Flagged: 98 (0.20% of users)
Top 3 Departments Contributing to Anomalies: HR, Operations, Legal


# Phase 6: Deployment

### Step 1: Model Serialization
Save the trained models and preprocessors to disk so they can be reused for new identities without retraining.

In [54]:
import joblib
import os

# Create a directory for models if it doesn't exist
models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)

# Note: In the previous cells, we used StandardScaler and TruncatedSVD but didn't save them to variables.
# Assuming they were created like: scaler = StandardScaler() and svd = TruncatedSVD(n_components=100)
# For completeness in deployment, we re-instantiate and fit them here or assume they exist.

try:
    # Save the models
    joblib.dump(kmeans_model, os.path.join(models_dir, 'kmeans_iam_model.pkl'))
    joblib.dump(dbscan_model, os.path.join(models_dir, 'dbscan_iam_model.pkl'))
    
    # Also save the scaler and SVD if they exist in your workspace
    if 'scaler' in locals():
        joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
    if 'svd' in locals():
        joblib.dump(svd, os.path.join(models_dir, 'svd_model.pkl'))
    if 'master_feature_matrix_df' in locals():
        joblib.dump(master_feature_matrix_df.columns, os.path.join(models_dir, 'model_columns.pkl'))
        
    print(f"Models successfully saved to the '{models_dir}' directory.")
except Exception as e:
    print(f"Error saving models: {e}")

Models successfully saved to the 'models' directory.


### Step 2: Exporting Results for BI/Reporting
Export the final enriched identity dataframe so it can be consumed by Dashboarding tools (Tableau, PowerBI, etc.)

In [55]:
# Create an output directory
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Export to CSV
csv_path = os.path.join(output_dir, 'identities_enriched_results.csv')
df_identities_enriched.to_csv(csv_path)
print(f"Results successfully exported to {csv_path}")

# Optionally export the anomalies separately for quick access
anomalies_csv_path = os.path.join(output_dir, 'high_risk_anomalies.csv')
df_anomalies.to_csv(anomalies_csv_path)
print(f"Anomalies specifically exported to {anomalies_csv_path}")

Results successfully exported to output\identities_enriched_results.csv
Anomalies specifically exported to output\high_risk_anomalies.csv


### Step 3: Example Inference Function (Pseudocode)
This illustrates how the saved models would be used in a production microservice when a new employee is onboarded.

In [59]:
import joblib
import os
import pandas as pd
import numpy as np

def predict_peer_group(new_user_features):
    """
    Demonstration function of how to use the saved models.
    
    Args:
        new_user_features (dict): A dictionary containing 'hr_features' and 'expected_access' arrays.
    """
    try:
        # 1. Load models & expected feature column names
        loaded_scaler = joblib.load(os.path.join('models', 'scaler.pkl'))
        loaded_svd = joblib.load(os.path.join('models', 'svd_model.pkl'))
        loaded_kmeans = joblib.load(os.path.join('models', 'kmeans_iam_model.pkl'))
        expected_columns = joblib.load(os.path.join('models', 'model_columns.pkl'))
        
        # 2. Preprocess payload into a single-row DataFrame
        row = {}
        
        # A. Map HR Features to One-Hot columns
        hr = new_user_features.get('hr_features', {})
        for k, v in hr.items():
            col_name = f"{k}_{v}"
            row[col_name] = 1  # Mark as presence (1)
            
        # B. Map Access Features to corresponding binary entitlement IDs
        access = new_user_features.get('expected_access', [])
        for ent_id in access:
            row[ent_id] = 1
            
        # C. Construct Aggregate Features
        row['total_entitlements_count'] = len(access)
        row['high_risk_entitlements_count'] = new_user_features.get('high_risk_count', 0)
        
        # Create DataFrame from the dictionary row
        df_new = pd.DataFrame([row])
        
        # Reindex to match the exact shape of master_feature_matrix_df, filling missing info with 0
        df_new = df_new.reindex(columns=expected_columns, fill_value=0)
        
        # Scale numerical aggregate columns using the saved Scaler
        agg_cols = ['total_entitlements_count', 'high_risk_entitlements_count']
        df_new[agg_cols] = loaded_scaler.transform(df_new[agg_cols])
        
        # Apply Dimensionality Reduction using the saved SVD
        features_reduced = loaded_svd.transform(df_new)
        
        # 3. Predict the cluster group
        predicted_group = loaded_kmeans.predict(features_reduced)
        return f"Successfully mapped to Peer Group: {predicted_group[0]}"
        
    except FileNotFoundError:
        return "Models not found. Ensure they are saved first by running Step 1."
    except Exception as e:
        return f"Error during prediction: {e}"

# --- Working Example ---
# Create a mock onboarding user that mimics our data structure
mock_new_user = {
    "identity_id": "test_service_account",
    "hr_features": {
        "department": "IT", 
        "job_title": "SysAdmin",
        "location": "Remote"
    },
    "expected_access": [
        "ENT_12345", "ENT_67890", "ENT_ABCDE"  # Examples of some dummy entitlements
    ],
    "high_risk_count": 1
}

print("Testing Inference Function:")
print(predict_peer_group(mock_new_user))


Testing Inference Function:
Successfully mapped to Peer Group: 6


In [64]:
import pandas as pd

# 1. Calculate Cluster Size
# We isolate DBSCAN anomalies (is_anomaly == True) to create a pure baseline 
valid_users = df_identities_enriched[df_identities_enriched['is_anomaly'] == False].reset_index()[['identity_id', 'kmeans_peer_group']].copy()
valid_users.rename(columns={'kmeans_peer_group': 'cluster_id'}, inplace=True)

# Total users in each distinct K-Means cluster
cluster_sizes = valid_users.groupby('cluster_id').size().reset_index(name='total_users')

# -------------------------------------------------------------------
# A. Confidence Score for Individual Entitlements
# -------------------------------------------------------------------

# Ensure unique identity-to-entitlement mappings
user_entitlements = valid_users.merge(account_entitlements_df[['identity_id', 'entitlement_id']], on='identity_id', how='inner').drop_duplicates()

# 2. Calculate Item Frequency (Entitlements)
ent_freq = user_entitlements.groupby(['cluster_id', 'entitlement_id']).size().reset_index(name='item_frequency')

# 3. Compute the Confidence Score (Entitlements)
ent_scores = ent_freq.merge(cluster_sizes, on='cluster_id')
ent_scores['confidence_score'] = (ent_scores['item_frequency'] / ent_scores['total_users']) * 100

# Map entitlement_id to actual entitlement_name
ent_scores = ent_scores.merge(entitlements_df[['entitlement_id', 'entitlement_name']], on='entitlement_id', how='left')
ent_scores.rename(columns={'entitlement_name': 'access_item_name'}, inplace=True)
ent_scores['item_type'] = 'entitlement'

# Map back to individual user
user_ent_scores = user_entitlements.merge(
    ent_scores[['cluster_id', 'entitlement_id', 'access_item_name', 'item_type', 'confidence_score']], 
    on=['cluster_id', 'entitlement_id'], 
    how='left'
)
user_ent_scores.drop(columns=['entitlement_id'], inplace=True)

# -------------------------------------------------------------------
# B. Confidence Score for Entitlement Groups
# -------------------------------------------------------------------

# Ensure unique identity-to-group mappings
user_groups = valid_users.merge(entitlement_group_assignments_df[['identity_id', 'ent_group_id']], on='identity_id', how='inner').drop_duplicates()

# 2. Calculate Item Frequency (Groups)
grp_freq = user_groups.groupby(['cluster_id', 'ent_group_id']).size().reset_index(name='item_frequency')

# 3. Compute the Confidence Score (Groups)
grp_scores = grp_freq.merge(cluster_sizes, on='cluster_id')
grp_scores['confidence_score'] = (grp_scores['item_frequency'] / grp_scores['total_users']) * 100

# Map ent_group_id to actual group_name
grp_scores = grp_scores.merge(entitlement_groups_df[['ent_group_id', 'group_name']], on='ent_group_id', how='left')
grp_scores.rename(columns={'group_name': 'access_item_name'}, inplace=True)
grp_scores['item_type'] = 'entitlement_group'

# Map back to individual user
user_grp_scores = user_groups.merge(
    grp_scores[['cluster_id', 'ent_group_id', 'access_item_name', 'item_type', 'confidence_score']], 
    on=['cluster_id', 'ent_group_id'], 
    how='left'
)
user_grp_scores.drop(columns=['ent_group_id'], inplace=True)

# -------------------------------------------------------------------
# 4. Map Back to the User (Final Output)
# -------------------------------------------------------------------
# Combine both entitlements and entitlement groups into one unpivoted dataframe
df_access_review_scores = pd.concat([user_ent_scores, user_grp_scores], ignore_index=True)

# Sort logically for output display
df_access_review_scores.sort_values(by=['cluster_id', 'confidence_score'], ascending=[True, False], inplace=True)
df_access_review_scores.reset_index(drop=True, inplace=True)

print("Sample of Peer Adoption Rate Confidence Scores:")
display(df_access_review_scores['confidence_score'].describe())


Sample of Peer Adoption Rate Confidence Scores:


count    1.140075e+06
mean     4.446447e+01
std      4.501065e+01
min      3.411805e-02
25%      3.552044e+00
50%      1.068567e+01
75%      1.000000e+02
max      1.000000e+02
Name: confidence_score, dtype: float64

In [ ]:
import plotly.express as px

fig = px.histogram(
    df_access_review_scores, 
    x="confidence_score", 
    nbins=20,
    title="Distribution of Confidence Scores across all Peer Groups",
    labels={'confidence_score': 'Confidence Score (%)', 'count':'Number of Unique Access Items'}
)
fig.show()
